# ML-02 — Research Question and Provisional Lane

This notebook frames my provisional capstone direction before any modeling.

## 1. My lane (or freestyle) and why

I am provisionally choosing **Lane 2: Refresh / Content Opportunity Scoring**. The practical problem is not simply identifying pages whose metrics moved down; it is deciding which pages deserve limited reviewer time first. This lane fits the starter dataset because it contains observable content, search, traffic, age, freshness, and trend signals at the content-item level. My intended output is a transparent ranked review queue with reason codes and confidence labels. I will compare a simple rule-based baseline with a more flexible scoring method only if the data shows that the added complexity is useful.

In [1]:
lane = {
    "name": "Refresh / Content Opportunity Scoring",
    "task_type": "ranking / scoring",
    "unit_of_analysis": "one pseudonymized content item (page)",
    "output": "ranked review queue with suggested actions, reason codes, and confidence labels",
}
print(lane)

{'name': 'Refresh / Content Opportunity Scoring', 'task_type': 'ranking / scoring', 'unit_of_analysis': 'one pseudonymized content item (page)', 'output': 'ranked review queue with suggested actions, reason codes, and confidence labels'}


## 2. The question: decision, action, cost of a wrong call

**Research question:** Using observable content and performance signals, which pages should a content reviewer inspect first for refresh, expansion, protection, pruning, or monitoring?

The decision is how to allocate a limited content-review budget across many pages. A content editor or SEO reviewer will act on the ranked queue by examining the highest-priority pages, checking their context, and then choosing an appropriate action rather than automatically changing them. A false positive can waste reviewer time or lead to an unnecessary edit that harms a useful page. A false negative can leave a meaningful decline or opportunity unnoticed. Missing a high-value page may be more costly than reviewing one extra page, but recommendations must remain explainable because edits can also cause harm.

Data can help combine volume, trend, position, engagement, age, and freshness consistently across thousands of pages. However, I will begin with a transparent rule baseline; ML is justified only if interacting signals improve the top of the review queue under honest validation. For a later future-window outcome, **precision@20** will match a workflow where a reviewer can inspect 20 pages, alongside recall and manual top-20 review.

In [2]:
decision_frame = {
    "decision": "Which pages should receive limited human review time first?",
    "actor_and_action": "A content reviewer inspects the top-ranked pages and chooses an action.",
    "false_positive_cost": "Wasted review time or an unnecessary, potentially harmful edit.",
    "false_negative_cost": "A meaningful decline or opportunity remains unnoticed.",
    "planned_primary_metric": "precision@20 against a future observed outcome",
}
print(decision_frame)

{'decision': 'Which pages should receive limited human review time first?', 'actor_and_action': 'A content reviewer inspects the top-ranked pages and chooses an action.', 'false_positive_cost': 'Wasted review time or an unnecessary, potentially harmful edit.', 'false_negative_cost': 'A meaningful decline or opportunity remains unnoticed.', 'planned_primary_metric': 'precision@20 against a future observed outcome'}


## 3. Quick look at the data (2–3 real numbers)

The following cell loads the repository's anonymized starter dataset. It checks three numbers that show why prioritization is worth investigating: the total number of content items, the number and share marked `down` by the starter trend proxy, and the number of declining items with at least 1,000 impressions in the trailing 90-day window. The 1,000-impression cutoff is an exploratory volume filter, not a universal definition of importance.

In [3]:
from pathlib import Path
import pandas as pd

data_path = Path("data/raw/content_refresh_anonymized.csv")
assert data_path.exists(), "Run this notebook from the repository root."

df = pd.read_csv(data_path)
is_down = df["trend_direction"].eq("down")
high_volume_down = is_down & df["impressions_90d"].ge(1_000)

evidence = pd.Series({
    "total_content_items": len(df),
    "declining_items": int(is_down.sum()),
    "declining_share_pct": round(is_down.mean() * 100, 1),
    "declining_items_with_1000plus_impressions": int(high_volume_down.sum()),
}, name="value")
print(evidence.to_frame())

                                             value
total_content_items                        30000.0
declining_items                            16262.0
declining_share_pct                           54.2
declining_items_with_1000plus_impressions   8031.0


### What these numbers suggest

The dataset contains **30,000** content items, and **16,262 (54.2%)** are marked `down` by the starter trend proxy. Even after applying an exploratory minimum of 1,000 impressions, **8,031** declining items remain. A reviewer cannot investigate all of these at once, so a ranked and explainable queue could improve the decision about where to look first. These figures establish scale and a prioritization need; they do not show that a refresh will cause recovery.

## 4. Careful words: what I can and can't claim

This project can report **observed associations** and provide **directional decision support** for human review. It may show that certain combinations of observable signals are useful for ranking pages against a clearly defined, later observed outcome. The current `trend_direction == 'down'` field is only a starter proxy calculated from the current window, not proof of future decline and not proof that a refresh is needed.

I cannot claim that any signal is a Google ranking factor, that a refresh caused or will cause recovery, or that a high-ranked page should be edited automatically. I also cannot claim causal impact without an experiment or suitable causal design. In later work I intend to replace or supplement the starter proxy with a leakage-safe future window—for example, prior-period features followed by a next-period decline or recovery outcome—and keep human review in the final decision.

In [4]:
claim_boundary = {
    "can_claim": ["observed patterns", "directional evidence", "decision-support ranking"],
    "cannot_claim": ["causal effect of refreshing", "Google ranking factors", "guaranteed recovery"],
}
print(claim_boundary)

{'can_claim': ['observed patterns', 'directional evidence', 'decision-support ranking'], 'cannot_claim': ['causal effect of refreshing', 'Google ranking factors', 'guaranteed recovery']}


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries appear anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — complete this after reviewing the executed notebook